# 그룹별로 나눠 보기

> 파이썬 6강 · 요약과 통계

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [그룹별로 나눠 보기](https://mioon1402.github.io/timeseriesdata/python/p06-groupby.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. groupby의 원리

**6-1. 요일별 평균 방문객**

In [ ]:
import pandas as pd
df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])

print(df.groupby("weekday")["visitors"].mean().round(1).to_string())

## 2. 순서 문제

**6-2. 순서 바로잡기**

In [ ]:
순서 = ["월", "화", "수", "목", "금", "토", "일"]

# 방법 1: reindex 로 원하는 순서를 강제한다 (간단)
결과 = df.groupby("weekday")["visitors"].mean().reindex(순서)
print(결과.round(1).to_string())

**6-3. 방법 2 — 순서형 범주로 지정**

In [ ]:
# 열 자체에 '순서'를 알려주면 이후 모든 작업에서 유지된다
df["weekday"] = pd.Categorical(df["weekday"], categories=순서, ordered=True)

print(df.groupby("weekday", observed=True)["visitors"].mean().round(1).to_string())
print()
print("정렬도 요일 순으로:", df.sort_values("weekday")["weekday"].head(3).tolist())

## 3. 여러 통계를 한 번에 — agg

**6-4. 요일별 종합 요약**

In [ ]:
요약 = df.groupby("weekday", observed=True).agg(
    일수     = ("visitors", "size"),
    평균방문 = ("visitors", "mean"),
    표준편차 = ("visitors", "std"),
    중앙값   = ("visitors", "median"),
    평균매출 = ("sales",    "mean"),
).round(1)

print(요약.to_string())

## 4. 두 개 이상의 기준으로 쪼개기

**6-5. 주말 여부 × 공휴일 여부**

In [ ]:
df["주말"] = df["date"].dt.dayofweek >= 5

결과 = df.groupby(["주말", "is_holiday"])["visitors"].agg(["size", "mean"]).round(1)
print(결과.to_string())

## 5. pivot_table — 표로 보기

**6-6. 연도 × 월 매출표**

In [ ]:
df["연"] = df["date"].dt.year
df["월"] = df["date"].dt.month

표 = df.pivot_table(values="sales", index="월", columns="연", aggfunc="mean")

print("월별 평균 매출 (만원)")
print((표 / 10000).round(1).to_string())

**6-7. 성장률 계산해보기**

In [ ]:
표 = df.pivot_table(values="sales", index="월", columns="연", aggfunc="mean")
표["성장률"] = (표[2025] / 표[2024] - 1) * 100

print((표[["성장률"]]).round(1).to_string())
print()
print(f"연간 평균 성장률: {표['성장률'].mean():.1f}%")

## 6. 그룹 크기를 꼭 함께 보세요

**6-8. 표본이 작으면 평균이 흔들린다**

In [ ]:
비교 = df.groupby("is_holiday").agg(
    일수     = ("visitors", "size"),
    평균     = ("visitors", "mean"),
    표준편차 = ("visitors", "std"),
)
# 표준오차 = 표준편차 / √n  (모듈 04)
비교["표준오차"] = 비교["표준편차"] / (비교["일수"] ** 0.5)
print(비교.round(1).to_string())

## 7. 실전: 쪼개면 방향이 바뀔 수도 있습니다

**6-9. 계절을 통제하고 비 효과 보기**

In [ ]:
def 계절(m):
    if m in (3, 4, 5):   return "봄"
    if m in (6, 7, 8):   return "여름"
    if m in (9, 10, 11): return "가을"
    return "겨울"

df["계절"] = df["월"].map(계절)
df["비"] = df["rain_mm"] > 0

# 전체로 볼 때
전체 = df.groupby("비")["visitors"].mean()
print(f"[전체]   비 안옴 {전체[False]:.1f}명 / 비 옴 {전체[True]:.1f}명  "
      f"→ 차이 {전체[True] - 전체[False]:+.1f}")

# 계절별로 나눠서 볼 때
print("\n[계절별]")
표 = df.pivot_table(values="visitors", index="계절", columns="비", aggfunc="mean")
표["차이"] = 표[True] - 표[False]
print(표.round(1).reindex(["봄", "여름", "가을", "겨울"]).to_string())

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 계절별 평균 매출과 일수를 함께 구해보세요.


# 문제 2. 요일별 '객단가'(매출 ÷ 방문객)를 구해보세요.
#        어느 요일이 가장 높은가요?


# 문제 3. 연도별·계절별 평균 매출을 pivot_table 로 만들어보세요.

**모범 답안**

In [ ]:
계절순 = ["봄", "여름", "가을", "겨울"]

# 문제 1
print(df.groupby("계절").agg(일수=("sales", "size"), 평균매출=("sales", "mean"))
        .reindex(계절순).round(0).to_string())

# 문제 2 — 주의: 각 날의 객단가를 먼저 구한 뒤 평균내야 한다
영업 = df[df["visitors"] > 0].copy()
영업["객단가"] = 영업["sales"] / 영업["visitors"]
print()
print(영업.groupby("weekday", observed=True)["객단가"].mean().round(0).to_string())

# 문제 3
print()
print((df.pivot_table(values="sales", index="계절", columns="연", aggfunc="mean")
         .reindex(계절순) / 10000).round(1).to_string())

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)